# IT Cost Forecasting — Notebook

Este notebook demonstra geração de dados sintéticos, modelagem simples e exportação de resultados para `reports/`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, sum as spark_sum, count as spark_count
from pyspark.sql.types import DoubleType
import random
import pandas as pd
import numpy as np
import logging
import os
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("it_cost_forecast")
spark = SparkSession.builder.getOrCreate()

In [ ]:
# Gera 24 meses de dados sintéticos para 5 cost centers
random.seed(42)
np.random.seed(42)
data = []
for month_offset in range(-24, 1):
    base_cost = 100000
    for center_id in range(1, 6):
        seasonal = 1 + 0.08 * np.sin(2 * np.pi * month_offset / 12)
        trend = 1 + 0.005 * abs(month_offset)
        noise = 1 + np.random.normal(0, 0.03)
        cost = base_cost * center_id * seasonal * trend * noise
        data.append({'month': f'2024-{13 + month_offset:02d}' if month_offset < 0 else f'2025-{month_offset + 1:02d}', 'cost_center': f'CC-{center_id:02d}', 'amount': round(cost, 2)})
df = spark.createDataFrame(data)
df = df.withColumn('month', to_date(col('month'), 'yyyy-MM'))
df = df.withColumn('amount', col('amount').cast(DoubleType()))
df = df.orderBy('month', 'cost_center')
print(f"Records: {df.count()}")

In [ ]:
# Data quality checks
from pyspark.sql.functions import count, when, min, max, avg, stddev
quality = df.select(count('*').alias('total_rows'), count(when(col('amount').isNull(), 1)).alias('null_amounts'), min('amount').alias('min_amount'), max('amount').alias('max_amount'), avg('amount').alias('avg_amount'))
quality.show()

In [ ]:
# Agrega para monthly totals
monthly = df.groupBy('month').agg(spark_sum('amount').alias('total_cost'), spark_count('cost_center').alias('num_centers')).orderBy('month')
monthly_pd = monthly.toPandas()
monthly_pd['month_num'] = range(len(monthly_pd))

In [ ]:
# Modelo simples: regressão linear em últimos 12 meses
from sklearn.linear_model import LinearRegression
recent = monthly_pd.tail(12).copy()
x = recent['month_num'].values.reshape(-1, 1)
y = recent['total_cost'].values
model = LinearRegression()
model.fit(x, y)
future_months = monthly_pd.tail(3)['month_num'].max() + np.arange(1, 4)
forecast = pd.DataFrame({'month': pd.date_range(start=monthly_pd['month'].max(), periods=4, freq='MS')[1:], 'forecast_cost': model.predict(future_months.reshape(-1, 1))})

In [ ]:
historical = monthly_pd[['month', 'total_cost']].copy()
historical.columns = ['month', 'cost']
historical['type'] = 'actual'
forecast_out = forecast.copy()
forecast_out.columns = ['month', 'cost']
forecast_out['type'] = 'forecast'
result = pd.concat([historical, forecast_out]).sort_values('month')
report_dir = os.path.join(os.path.dirname(__file__), '..', '..', 'reports')
os.makedirs(report_dir, exist_ok=True)
result_path = os.path.join(report_dir, 'forecast_result.csv')
result.to_csv(result_path, index=False)
logger.info("Forecast result written to %s", result_path)
print(result.to_string(index=False))

In [ ]:
# Validação e métricas
validation = monthly_pd.tail(6).copy()
validation['predicted'] = model.predict(validation['month_num'].values.reshape(-1, 1))
validation['error_pct'] = ((validation['total_cost'] - validation['predicted']) / validation['total_cost'] * 100).round(2)
validation_path = os.path.join(report_dir, 'forecast_validation.csv')
validation.to_csv(validation_path, index=False)
logger.info("Validation CSV written to %s", validation_path)
mean_error = validation['error_pct'].abs().mean()
print(f"Mean absolute error: {mean_error:.1f}%")
print(f"R² on recent data: {model.score(x, y):.4f}")

In [ ]:
def run_forecast(output_dir: str = None):
    if output_dir is None:
        output_dir = report_dir
    logger.info("Forecast run complete. Reports available in %s", output_dir)
    return {"forecast_csv": result_path, "validation_csv": validation_path, "mean_error_pct": mean_error, "r2": model.score(x, y)}

if __name__ == '__main__':
    out = run_forecast()
    print(out)